[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natrask/AESCAPE/blob/main/notebooks/06_agent_hackathon.ipynb)

# Part 6 — Agent-driven adaptive mesh refinement

Parts 1 to 5 built agent patterns on problems small enough to read at a glance. This notebook points one at a real numerical problem.

The problem is Poisson's equation on an L-shaped domain:

$$-\nabla^2 u = 1 \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

The re-entrant corner makes the solution singular. Uniform refinement therefore converges slowly, while refining only where the error actually lives recovers the optimal rate. The agent's job is to find that out by driving the mesh itself.

We use the **ReAct** loop from Part 1, unchanged. Nothing from Parts 2 to 5 is needed: the number of refinement cycles depends on error nobody has measured yet, and a bad refinement only costs a mesh we throw away, which is the case ReAct handles well.

**What does this notebook do?** We hand an agent seven tools — describe a mesh, solve, summarise the local error, refine uniformly, refine adaptively, check the convergence rate, and stop — then run three agents that differ only in their system prompts. A separate verifier agent checks the result with a manufactured solution.

**You are done when** the adaptive agent reaches the error tolerance using fewer degrees of freedom than the uniform one, and the convergence plot shows the two slopes separating.

## 6.0 Setup

The same setup block as Parts 1 to 5, plus `scikit-fem` for the solver and `sympy` for the verifier. It also defines `run_agent`: the ReAct loop from Part 1, unchanged.

In [ ]:
# Setup -- same as 00_api_access.ipynb, plus the FEM stack and run_agent from Part 1.
import os, sys, time, json
if 'google.colab' in sys.modules:
    %pip install -U -q "google-genai<2.13" "google-auth==2.49.0" scikit-fem sympy

import numpy as np
import matplotlib.pyplot as plt

from google import genai
from google.genai import types as gtypes

from skfem import (MeshTri, Basis, ElementTriP1, BilinearForm, LinearForm,
                   Functional, condense, solve)
from skfem.helpers import dot, grad

plt.rcParams['figure.dpi'] = 90

try:
    from google.colab import userdata
    API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    API_KEY = os.environ.get('GEMINI_API_KEY', '')

assert API_KEY, "Set GEMINI_API_KEY in Colab secrets or your environment."

client = genai.Client(api_key=API_KEY)
MODEL = "gemini-3.1-flash-lite"


def generate_with_retry(*, contents, config=None, max_attempts=6):
    delay = 4.0
    for attempt in range(max_attempts):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=config)
        except Exception as e:
            msg = str(e)
            if "429" not in msg and "RESOURCE_EXHAUSTED" not in msg and "quota" not in msg.lower():
                raise
            if attempt == max_attempts - 1:
                raise
            print(f"[rate limit] attempt {attempt+1}: sleeping {delay:.1f}s and retrying...")
            time.sleep(delay)
            delay = min(delay * 2, 60.0)


def run_agent(system_prompt, user_prompt, tool_schemas, tool_fns,
              max_steps=12, temperature=0.2, verbose=True):
    """Minimal ReAct loop. Returns (final_text, transcript)."""
    contents = [gtypes.Content(role="user",
                               parts=[gtypes.Part.from_text(text=user_prompt)])]
    cfg = gtypes.GenerateContentConfig(
        system_instruction=system_prompt,
        tools=[gtypes.Tool(function_declarations=tool_schemas)],
        temperature=temperature,
        automatic_function_calling=gtypes.AutomaticFunctionCallingConfig(disable=True))

    transcript = []
    for step in range(max_steps):
        resp = generate_with_retry(contents=contents, config=cfg)
        parts = resp.candidates[0].content.parts or []
        contents.append(resp.candidates[0].content)

        calls = [p.function_call for p in parts if getattr(p, "function_call", None)]
        if not calls:
            text = "".join(getattr(p, "text", "") or "" for p in parts)
            transcript.append(("final", text))
            if verbose: print(f"[{step}] FINAL: {text[:200]}")
            return text, transcript

        obs = []
        for fc in calls:
            name, args = fc.name, dict(fc.args or {})
            if verbose: print(f"[{step}] CALL {name}({args})")
            try:
                result = tool_fns[name](**args)
            except Exception as e:
                result = {"error": f"{type(e).__name__}: {e}"}
            transcript.append((name, args, result))
            if verbose: print(f"[{step}]   -> {json.dumps(result)[:200]}")
            obs.append(gtypes.Part.from_function_response(name=name, response=result))
        contents.append(gtypes.Content(role="user", parts=obs))

    transcript.append(("final", "(max_steps reached)"))
    return "(max_steps reached)", transcript

print(f"Gemini client ready (model={MODEL}).")

## 6.1 The problem

The Poisson problem $-\Delta u = 1$ with $u=0$ on the boundary of an L-shaped domain has a re-entrant corner at the origin. The solution behaves locally like $r^{2/3}\sin(2\theta/3)$, so $u \in H^{5/3-\epsilon}$ but $u \notin H^2$.

Consequences for P1 finite elements:

- Uniform refinement gives the suboptimal asymptotic rate $O(h^{2/3})$ in the $H^1$-seminorm. At the mesh sizes used here, pre-asymptotic rates of 0.7 to 0.95 are typical.
- Adaptive refinement driven by a local error indicator recovers the optimal rate $O(N^{-1/2})$, i.e. rate 1 against $h \sim N^{-1/2}$.

In this part we give an LLM a set of mesh-manipulation tools and let it drive the refinement. Three variants of agent (A1, A2, A3), then a separate verifier agent that runs a manufactured-solution test.

For a refresher on the scikit-fem API used here, see the [scikit-fem documentation](https://scikit-fem.readthedocs.io/en/latest/listofexamples.html) - in particular the Poisson examples.

## 6.2 Forms and the solver

`solve_poisson_on(mesh)` solves $-\Delta u = 1$ with $u=0$ on the boundary and returns the solution vector and basis.

In [ ]:
@BilinearForm
def stiffness(u, v, w):
    return dot(grad(u), grad(v))

@LinearForm
def unit_load(v, w):
    return 1.0 * v

def solve_poisson_on(mesh):
    basis = Basis(mesh, ElementTriP1())
    K = stiffness.assemble(basis)
    b = unit_load.assemble(basis)
    u = solve(*condense(K, b, D=mesh.boundary_nodes()))
    return u, basis

def energy_norm_sq(mesh, u):
    basis = Basis(mesh, ElementTriP1())
    @Functional
    def grad_sq(w): return w.w.grad[0]**2 + w.w.grad[1]**2
    return grad_sq.assemble(basis, w=basis.interpolate(u))

m0 = MeshTri.init_lshaped()
fig, ax = plt.subplots(figsize=(5, 5))
ax.triplot(m0.p[0], m0.p[1], m0.t.T, lw=0.5)
ax.set_aspect('equal'); ax.set_title(f"Initial L-shape mesh ({m0.nelements} triangles)")
plt.show()

## 6.3 Reference energy

The true solution is not closed-form on the L-shape. Galerkin orthogonality gives
$$
\|u - u_h\|_E^2 \;=\; a(u,u) - a(u_h, u_h),
$$
where $a(\cdot,\cdot) = \int \nabla u \cdot \nabla v$. Take $u$ as the solution on a very fine uniform mesh, treat $a(u,u)$ as the reference, and compute energy-error estimates for any coarser solve.

In [ ]:
print("Computing reference energy (one-time setup)...", flush=True)
m_ref = MeshTri.init_lshaped()
for _ in range(7):
    m_ref = m_ref.refined()
u_ref, basis_ref = solve_poisson_on(m_ref)
E_REF = energy_norm_sq(m_ref, u_ref)
print(f"Reference: {basis_ref.N} dofs, energy = {E_REF:.6f}")

def h1_error(mesh, u):
    return float(np.sqrt(max(E_REF - energy_norm_sq(mesh, u), 0.0)))

## 6.4 Error indicator and Dörfler marking

Gradient-jump indicator on interior edges:
$$
\eta_K^2 \;=\; \tfrac{1}{2}\sum_{e \in \partial K \cap \Omega^\circ} h_e\, |[\![ \nabla u_h \cdot n_e ]\!]|^2.
$$

Dorfler marking picks the smallest set of elements whose squared indicators sum to at least a fraction $\theta$ of the total. We use $\theta = 0.5$.

In [ ]:
def gradient_jump_indicator(mesh, u):
    p, t, facets, f2t = mesh.p, mesh.t, mesh.facets, mesh.f2t
    nelem = t.shape[1]
    g = np.zeros((2, nelem))
    for k in range(nelem):
        i0, i1, i2 = t[:, k]
        A = np.column_stack([p[:, i1] - p[:, i0], p[:, i2] - p[:, i0]])
        rhs = np.array([u[i1] - u[i0], u[i2] - u[i0]])
        g[:, k] = np.linalg.solve(A.T, rhs)
    eta_sq = np.zeros(nelem)
    for e in range(facets.shape[1]):
        tp, tm = f2t[0, e], f2t[1, e]
        if tm < 0: continue
        i0, i1 = facets[:, e]
        edge = p[:, i1] - p[:, i0]
        h_e = np.linalg.norm(edge)
        n = np.array([edge[1], -edge[0]]) / h_e
        jump = (g[:, tp] - g[:, tm]) @ n
        contrib = h_e * jump**2
        eta_sq[tp] += 0.5 * contrib
        eta_sq[tm] += 0.5 * contrib
    return np.sqrt(eta_sq)

def dorfler_mark(indicators, theta=0.5):
    eta2 = indicators**2
    idx = np.argsort(-eta2)
    cum = np.cumsum(eta2[idx])
    k = int(np.searchsorted(cum, theta * cum[-1]) + 1)
    return np.sort(idx[:k])

## 6.5 Tools for the FEM agent

The agent can't pass Python mesh objects through JSON, so we keep a mesh registry keyed by integer IDs. Each tool takes a `mesh_id`, performs its work, and returns a dict.

Two guardrails:
- `MAX_DOFS = 20000` caps the final mesh size.
- Every tool is wrapped in `try/except` so the agent sees errors as observations.

In [ ]:
MAX_DOFS = 20000
MESH_REGISTRY = {}
NEXT_ID = [0]
HISTORY = []

def _register(mesh):
    mid = NEXT_ID[0]; NEXT_ID[0] += 1
    MESH_REGISTRY[mid] = mesh
    return mid

def reset_fem_state():
    MESH_REGISTRY.clear()
    NEXT_ID[0] = 0
    HISTORY.clear()
    m0 = MeshTri.init_lshaped()
    return _register(m0)

def tool_describe_mesh(mesh_id: int):
    m = MESH_REGISTRY.get(mesh_id)
    if m is None: return {"error": f"no mesh with id {mesh_id}"}
    return {"mesh_id": mesh_id, "vertices": int(m.p.shape[1]), "elements": int(m.nelements)}

def tool_solve_poisson(mesh_id: int):
    m = MESH_REGISTRY.get(mesh_id)
    if m is None: return {"error": f"no mesh with id {mesh_id}"}
    u, basis = solve_poisson_on(m)
    err = h1_error(m, u)
    step = len(HISTORY)
    HISTORY.append({"step": step, "mesh_id": mesh_id, "dofs": int(basis.N), "H1_error": err})
    MESH_REGISTRY[mesh_id] = m
    MESH_REGISTRY[(mesh_id, "u")] = u
    return {"mesh_id": mesh_id, "dofs": int(basis.N), "H1_error": err, "step": step}

def tool_local_indicators(mesh_id: int):
    m = MESH_REGISTRY.get(mesh_id)
    u = MESH_REGISTRY.get((mesh_id, "u"))
    if m is None or u is None:
        return {"error": f"call solve_poisson({mesh_id}) first"}
    eta = gradient_jump_indicator(m, u)
    return {
        "num_elements": int(len(eta)),
        "eta_max":   float(eta.max()),
        "eta_mean":  float(eta.mean()),
        "eta_min":   float(eta.min()),
    }

def tool_uniform_refine(mesh_id: int):
    m = MESH_REGISTRY.get(mesh_id)
    if m is None: return {"error": f"no mesh with id {mesh_id}"}
    m_new = m.refined()
    if m_new.p.shape[1] > MAX_DOFS:
        return {"error": f"refinement would produce {m_new.p.shape[1]} dofs > cap {MAX_DOFS}"}
    new_id = _register(m_new)
    return {"new_mesh_id": new_id, "new_vertices": int(m_new.p.shape[1]),
            "new_elements": int(m_new.nelements)}

def tool_refine_by_threshold(mesh_id: int, theta: float = 0.5):
    """Dorfler marking with bulk fraction theta, then refine."""
    m = MESH_REGISTRY.get(mesh_id)
    u = MESH_REGISTRY.get((mesh_id, "u"))
    if m is None or u is None:
        return {"error": f"call solve_poisson({mesh_id}) first"}
    if not (0 < theta < 1):
        return {"error": f"theta must be in (0,1), got {theta}"}
    eta = gradient_jump_indicator(m, u)
    marked = dorfler_mark(eta, theta=theta)
    m_new = m.refined(marked)
    if m_new.p.shape[1] > MAX_DOFS:
        return {"error": f"refinement would produce {m_new.p.shape[1]} dofs > cap {MAX_DOFS}"}
    new_id = _register(m_new)
    return {"new_mesh_id": new_id, "marked": int(len(marked)),
            "new_vertices": int(m_new.p.shape[1]),
            "new_elements": int(m_new.nelements)}

def tool_check_rate(mesh_id: int = -1):
    """Return empirical log-log slope from the last 3 solves."""
    if len(HISTORY) < 3:
        return {"error": "need at least 3 solves first"}
    h = np.array([1.0/np.sqrt(r["dofs"]) for r in HISTORY[-3:]])
    err = np.array([r["H1_error"] for r in HISTORY[-3:]])
    slope = float(np.polyfit(np.log(h), np.log(err), 1)[0])
    interp = ("close to adaptive optimum (1.0)" if slope > 0.9
              else f"rate {slope:.3f} below adaptive optimum (1.0); uniform predicts 2/3")
    return {"empirical_rate": slope, "interpretation": interp,
            "last3_dofs": [int(r["dofs"]) for r in HISTORY[-3:]],
            "last3_errors": [float(r["H1_error"]) for r in HISTORY[-3:]]}

def tool_stop(reason: str = ""):
    return {"stopped": True, "reason": reason}

FEM_TOOL_FNS = {
    "describe_mesh":        tool_describe_mesh,
    "solve_poisson":        tool_solve_poisson,
    "local_indicators":     tool_local_indicators,
    "uniform_refine":       tool_uniform_refine,
    "refine_by_threshold":  tool_refine_by_threshold,
    "check_convergence_rate": tool_check_rate,
    "stop":                 tool_stop,
}

FEM_TOOL_SCHEMAS = [
    gtypes.FunctionDeclaration(name="describe_mesh", description="Report vertex and element counts for a mesh.",
        parameters={"type":"object","properties":{"mesh_id":{"type":"integer"}},"required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="solve_poisson", description="Solve -Laplace(u)=1, u=0 on boundary. Returns dofs and H1-seminorm error estimate.",
        parameters={"type":"object","properties":{"mesh_id":{"type":"integer"}},"required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="local_indicators", description="Summary statistics of the per-element gradient-jump indicators on the current mesh.",
        parameters={"type":"object","properties":{"mesh_id":{"type":"integer"}},"required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="uniform_refine", description="Uniformly refine every element once. Returns new_mesh_id.",
        parameters={"type":"object","properties":{"mesh_id":{"type":"integer"}},"required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="refine_by_threshold", description="Dorfler-mark the highest-indicator elements (bulk fraction theta in (0,1)) and refine them. Returns new_mesh_id.",
        parameters={"type":"object","properties":{
            "mesh_id":{"type":"integer"},
            "theta":{"type":"number","description":"bulk fraction; 0.5 is standard"}},
            "required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="check_convergence_rate", description="Empirical log-log slope of H1-error vs h over the last 3 solves.",
        parameters={"type":"object","properties":{"mesh_id":{"type":"integer"}},"required":[]}),
    gtypes.FunctionDeclaration(name="stop", description="Signal that the refinement loop is done.",
        parameters={"type":"object","properties":{"reason":{"type":"string"}},"required":[]}),
]
print(f"Registered {len(FEM_TOOL_FNS)} FEM tools.")

## 6.6 A1: uniform refinement

The first agent refines every element on every cycle. Its prompt tells it to loop `solve_poisson` and `uniform_refine` until the H1 error falls below 5e-2, then call `stop`.

This is the baseline. On a domain with a re-entrant corner uniform refinement converges at the suboptimal rate of about 2/3, so watch how fast the degree-of-freedom count climbs against how slowly the error falls.

In [ ]:
reset_fem_state()
initial_mesh_id = 0

A1_SYSTEM = """You drive a finite element mesh toward a target accuracy.

Repeat until you stop:
  1. Call solve_poisson(mesh_id) and read H1_error.
  2. If H1_error < 5e-2, call stop and report the final mesh_id and its dofs.
  3. Otherwise call uniform_refine(mesh_id) and continue with the new_mesh_id it returns.

Use uniform_refine only; do not call refine_by_threshold. If a refinement is refused
because it would exceed the degree-of-freedom cap, call stop and say so."""

A1_USER = f"Starting mesh_id={initial_mesh_id}. Target H1_error < 5e-2."

answer_A1, transcript_A1 = run_agent(
    system_prompt=A1_SYSTEM,
    user_prompt=A1_USER,
    tool_schemas=FEM_TOOL_SCHEMAS,
    tool_fns=FEM_TOOL_FNS,
    max_steps=14,
    verbose=True,
)
HISTORY_A1 = list(HISTORY)

In [ ]:
def plot_convergence(histories, labels, theory_slopes=None):
    fig, ax = plt.subplots(figsize=(7, 5))
    for hist, lab in zip(histories, labels):
        if not hist: continue
        h = np.array([1.0/np.sqrt(r['dofs']) for r in hist])
        e = np.array([r['H1_error']          for r in hist])
        ax.loglog(h, e, 'o-', label=f"{lab}  (final err={e[-1]:.2e})")
    if theory_slopes:
        h_ref = np.array([0.4, 0.01])
        for slope, lab in theory_slopes:
            ax.loglog(h_ref, 0.5 * (h_ref/0.4)**slope, '--', alpha=0.6, label=f"slope {slope:.2f} ({lab})")
    ax.set_xlabel("h = 1/sqrt(DOFs)"); ax.set_ylabel("H1 error estimate")
    ax.set_title("Convergence"); ax.grid(True, which='both', alpha=0.3); ax.legend()
    ax.invert_xaxis()
    plt.tight_layout(); plt.show()

plot_convergence([HISTORY_A1], ["A1 uniform"],
                 theory_slopes=[(2/3, "theory uniform"), (1.0, "theory adaptive")])

## 6.7 A2: adaptive refinement

The second agent has the same target but uses `refine_by_threshold(mesh_id, theta)`, which refines only the elements carrying most of the estimated error.

`theta` is the Dörfler bulk fraction. At 0.5 the tool marks the smallest set of elements holding half of the total squared indicator. The expected payoff is the optimal rate of 1, reached with far fewer degrees of freedom than A1 needed.

In [ ]:
reset_fem_state()
initial_mesh_id = 0

A2_SYSTEM = """You drive a finite element mesh toward a target accuracy, refining only where
the error is concentrated.

Repeat until you stop:
  1. Call solve_poisson(mesh_id) and read H1_error.
  2. If H1_error < 5e-2, call stop and report the final mesh_id and its dofs.
  3. Otherwise call refine_by_threshold(mesh_id, theta=0.5) and continue with the
     new_mesh_id it returns.

theta is the Dorfler bulk fraction: 0.5 marks the smallest set of elements carrying half of
the total squared error. Use refine_by_threshold, not uniform_refine. If a refinement is
refused because it would exceed the degree-of-freedom cap, call stop and say so."""

A2_USER = f"Starting mesh_id={initial_mesh_id}. Target H1_error < 5e-2."

answer_A2, transcript_A2 = run_agent(
    system_prompt=A2_SYSTEM, user_prompt=A2_USER,
    tool_schemas=FEM_TOOL_SCHEMAS, tool_fns=FEM_TOOL_FNS,
    max_steps=14, verbose=True,
)
HISTORY_A2 = list(HISTORY)

In [ ]:
# Compare A1 and A2 and show the final adaptive mesh.
final_mesh_id_A2 = max(i for i in MESH_REGISTRY if isinstance(i, int))
final_mesh = MESH_REGISTRY[final_mesh_id_A2]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.triplot(final_mesh.p[0], final_mesh.p[1], final_mesh.t.T, lw=0.3, color='k')
ax.set_aspect('equal'); ax.set_title(f"A2 final adaptive mesh ({final_mesh.nelements} elems)")

ax = axes[1]
for hist, lab in [(HISTORY_A1, "A1 uniform"), (HISTORY_A2, "A2 adaptive")]:
    if not hist: continue
    h = np.array([1.0/np.sqrt(r['dofs']) for r in hist])
    e = np.array([r['H1_error']          for r in hist])
    ax.loglog(h, e, 'o-', label=f"{lab}  (final err={e[-1]:.2e})")
h_ref = np.array([0.4, 0.01])
for slope, lab in [(2/3, "theory uniform"), (1.0, "theory adaptive")]:
    ax.loglog(h_ref, 0.5 * (h_ref/0.4)**slope, '--', alpha=0.6, label=f"slope {slope:.2f} ({lab})")
ax.set_xlabel("h = 1/sqrt(DOFs)"); ax.set_ylabel("H1 error estimate")
ax.set_title("Convergence"); ax.grid(True, which='both', alpha=0.3); ax.legend()
ax.invert_xaxis()
plt.tight_layout(); plt.show()

## 6.8 A3: adaptive refinement with rate checking

The third agent adds `check_convergence_rate`, which reports the log-log slope of the last three solves together with a short comparison to theory. Its prompt tells it to check the rate every cycle and raise `theta` from 0.5 to 0.7 if the rate drops below 0.9 once the mesh is past 500 degrees of freedom.

A larger `theta` marks more elements per cycle, so the mesh grows faster. The tool needs three solves before it can report a rate, so the first two cycles return an error instead and the prompt tells the agent to carry on.

In [ ]:
reset_fem_state()
initial_mesh_id = 0

A3_SYSTEM = """You drive a finite element mesh adaptively and watch how fast it converges.

Start with theta = 0.5 and repeat until you stop:
  1. Call solve_poisson(mesh_id) and read H1_error and dofs.
  2. If H1_error < 5e-2, call stop and report the final mesh_id and its dofs.
  3. Call check_convergence_rate() and read empirical_rate.
  4. If dofs is above 500 and empirical_rate is below 0.9, set theta = 0.7 and keep it
     there for the rest of the run. Otherwise leave theta unchanged.
  5. Call refine_by_threshold(mesh_id, theta) and continue with the new_mesh_id it returns.

A larger theta marks more elements per cycle. check_convergence_rate needs at least three
solves before it can report a rate; if it returns an error, skip step 4 for that cycle and
carry on. If a refinement is refused because it would exceed the degree-of-freedom cap,
call stop and say so."""

A3_USER = f"Starting mesh_id={initial_mesh_id}. Adaptive refinement with rate checking."

answer_A3, transcript_A3 = run_agent(
    system_prompt=A3_SYSTEM, user_prompt=A3_USER,
    tool_schemas=FEM_TOOL_SCHEMAS, tool_fns=FEM_TOOL_FNS,
    max_steps=18, verbose=True,
)
HISTORY_A3 = list(HISTORY)

In [ ]:
plot_convergence(
    [HISTORY_A1, HISTORY_A2, HISTORY_A3],
    ["A1 uniform", "A2 adaptive", "A3 rate-checked"],
    theory_slopes=[(2/3, "uniform"), (1.0, "adaptive")]
)

## 6.9 Verifier agent: manufactured-solution test

The estimates above use an energy extrapolation that assumes `solve_poisson_on` is correct. As an independent check we run a manufactured-solution test on the final mesh.

Pick a smooth $u_{\text{exact}}(x,y)$, compute $f = -\Delta u_{\text{exact}}$, solve $-\Delta u_h = f$ with $u_h = u_{\text{exact}}$ on the boundary, and measure $\|u_h - u_{\text{exact}}\|$. For smooth $u_{\text{exact}}$ and P1 elements the expected rates are $L^2$ error $O(h^2)$ and $H^1$-seminorm error $O(h)$.

A separate agent chooses $u_{\text{exact}}$ and runs the test.

In [ ]:
def tool_manufactured_test(mesh_id: int, u_expr: str = "sin(pi*x)*sin(pi*y)"):
    """Run a manufactured-solution test on the given mesh.

    u_expr: Python expression in x, y using numpy functions and pi.
    Returns L2 and H1-seminorm errors vs the exact u_expr.
    """
    m = MESH_REGISTRY.get(mesh_id)
    if m is None: return {"error": f"no mesh with id {mesh_id}"}

    import sympy as sp
    xs, ys = sp.symbols('x y')
    u_sym = sp.sympify(u_expr, locals={'pi': sp.pi})
    lap_sym = sp.diff(u_sym, xs, 2) + sp.diff(u_sym, ys, 2)
    f_sym = -lap_sym
    dudx_sym = sp.diff(u_sym, xs)
    dudy_sym = sp.diff(u_sym, ys)
    f_fn    = sp.lambdify((xs, ys), f_sym,    'numpy')
    u_fn    = sp.lambdify((xs, ys), u_sym,    'numpy')
    dudx_fn = sp.lambdify((xs, ys), dudx_sym, 'numpy')
    dudy_fn = sp.lambdify((xs, ys), dudy_sym, 'numpy')

    @LinearForm
    def manufactured_load(v, w):
        return f_fn(w.x[0], w.x[1]) * v

    basis = Basis(m, ElementTriP1())
    K = stiffness.assemble(basis)
    b = manufactured_load.assemble(basis)
    bnd = m.boundary_nodes()
    u_D = np.zeros(basis.N)
    u_D[bnd] = u_fn(m.p[0, bnd], m.p[1, bnd])
    u = solve(*condense(K, b, D=bnd, x=u_D))

    @Functional
    def L2_sq(w):  return (w.w.value - u_fn(w.x[0], w.x[1]))**2
    @Functional
    def H1_sq(w):
        return (w.w.grad[0] - dudx_fn(w.x[0], w.x[1]))**2 + \
               (w.w.grad[1] - dudy_fn(w.x[0], w.x[1]))**2
    w = basis.interpolate(u)
    L2 = float(np.sqrt(L2_sq.assemble(basis, w=w)))
    H1 = float(np.sqrt(H1_sq.assemble(basis, w=w)))
    return {"u_exact": u_expr, "dofs": int(basis.N),
            "L2_error": L2, "H1_semi_error": H1,
            "expected_rates": "L2: O(h^2), H1: O(h^1) for smooth u and P1 elements"}

VERIFIER_TOOL_FNS = {"manufactured_test": tool_manufactured_test, "stop": tool_stop}
VERIFIER_TOOL_SCHEMAS = [
    gtypes.FunctionDeclaration(name="manufactured_test",
        description="Run a manufactured-solution test: choose u_exact(x,y), compute f = -Laplacian(u_exact), solve, report L2 and H1 errors.",
        parameters={"type":"object","properties":{
            "mesh_id":{"type":"integer"},
            "u_expr":{"type":"string","description":"expression in x, y, e.g. 'sin(pi*x/2)*sin(pi*y/2)'"}},
            "required":["mesh_id"]}),
    gtypes.FunctionDeclaration(name="stop", description="Report the verification result and finish.",
        parameters={"type":"object","properties":{"reason":{"type":"string"}},"required":[]}),
]

In [ ]:
candidate_id = max(i for i in MESH_REGISTRY if isinstance(i, int))

VERIFIER_SYSTEM = """You are a verification agent. Another procedure produced a Poisson
solve on mesh_id={mid}. Design a manufactured-solution test (pick a smooth u_exact
that is differentiable everywhere and produces a nontrivial forcing), run
manufactured_test, and judge whether the observed L2 and H1 errors are consistent
with the expected rates for P1 elements. Report PASS or FAIL with a one-sentence
justification.""".format(mid=candidate_id)

VERIFIER_USER = f"Verify mesh_id={candidate_id}."

verifier_answer, verifier_transcript = run_agent(
    system_prompt=VERIFIER_SYSTEM, user_prompt=VERIFIER_USER,
    tool_schemas=VERIFIER_TOOL_SCHEMAS, tool_fns=VERIFIER_TOOL_FNS,
    max_steps=6, verbose=True,
)
print("\n=== Verifier verdict:")
print(verifier_answer)

## 6.10 Further reading

**Adaptive finite elements**

- Dörfler, *A convergent adaptive algorithm for Poisson's equation*, SIAM J. Numer. Anal. 33(3), 1996 — the marking strategy used here.
- [scikit-fem documentation](https://scikit-fem.readthedocs.io/) and its [example gallery](https://scikit-fem.readthedocs.io/en/latest/listofexamples.html).

**Agents doing science**

- [FunSearch](https://www.nature.com/articles/s41586-023-06924-6) (*Nature* 625, 2024) and [AlphaEvolve](https://deepmind.google/discover/blog/alphaevolve-a-gemini-powered-coding-agent-for-designing-advanced-algorithms/) — LLM-guided program search.
- [ChemCrow](https://arxiv.org/abs/2304.05376) and [Coscientist](https://www.nature.com/articles/s41586-023-06792-0) — agents driving chemistry experiments.
- [Sakana AI Scientist](https://sakana.ai/ai-scientist/) — an autonomous ML research loop.